This notebook calculates a magnetic local time dependent Pc5 ULF index from yearly SuperMAG data using wavelet transform <br>
There is a script for writing the data into a file

In [1]:
import numpy as np
import pandas as pd
import datetime
import pycwt as wavelet
from pycwt.helpers import find
from netCDF4 import Dataset

In [2]:
# function that calculates and returns the horizontal component of the magnetic field
def horizontal(df):
    Bn = df['dbn_nez'].to_numpy()
    Be = df['dbe_nez'].to_numpy()
    Bh = np.sqrt(Bn**2+Be**2)
    return Bh

#function that calculates and returns the north component of the magnetic field
def north(df):
    Bn = df['dbn_nez'].to_numpy()
    return Bn

# function that calculates and returns the east component of the magnetic field
def east(df):
    b = df['dbe_nez'].to_numpy()
    return b

In [9]:
# function that does the wavelet transform and calculcates the scale averaged Pc5 power 
# and returns the wavelet power, frequencies, scales, scale-averaged Pc5 power and cone of influence

def pc5_wave(dat, dt):
   mother = wavelet.Morlet(6) # mother wavelet 
   s0 = 2 * dt  # Starting scale, 2 * 60 s = 2 mins
   dj = 1 / 12  # Twelve sub-octaves per octaves
   J = 9 / dj  # Nine powers of two with dj sub-octaves

   # wavelet transform
   wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat, dt, dj, s0, J,mother)

   power = (np.abs(wave)) ** 2

   period = 1 / freqs

   # select the Pc5 period (150s – 600s)
   sel1 = find((period >= 150) & (period < 600))

   # scale average as in Torrence and Compo (1998) equation 24
   Cdelta = mother.cdelta
   scale_avg = (scales * np.ones((np.shape(power)[1], 1))).transpose() 
   scale_avg = power / scale_avg 
   pc5_scale_avg = dj * dt / Cdelta * scale_avg[sel1, :].sum(axis=0)

   return power, freqs, scales, pc5_scale_avg, coi

Change the parameters in the following cell 

In [4]:
# set properties

# year
YYYY = 2017

# latitude (60-70,70-80,etc.)
lat_set = (65,70)

# component (N, E or H)
component = 'N'

In [5]:
# component selection things

component_dict = {'H':False,'N':False,'E':False}
component_funcs = {'H':horizontal,'N':north,'E':east}
component_dict[component] = True

In [6]:
filename = f'all_stations_all{YYYY}.netcdf'

In [7]:
# read netCDF data
f = Dataset(filename)

# get the needed variables
Bn = f.variables['dbn_nez'][:] #magnetic field north component
Be = f.variables['dbe_nez'][:] #magnetic field east component

yr = f.variables['time_yr'][:] #year
mo = f.variables['time_mo'][:] #months
day = f.variables['time_dy'][:] #days
hr = f.variables['time_hr'][:] #hours
mt = f.variables['time_mt'][:] #minutes
mlat = f.variables['mlat'][:] #magnetic latitudes
mlt = f.variables['mlt'][:] #magnetic local time

# make dates into datetime object
date_UTC = [datetime.datetime(year=yr[i], month=mo[i], day=day[i],hour=hr[i],minute=mt[i]) for i in range(len(mo))]

# separation into latitudes

# select stations between selected magnetic latitude range
s6070 = np.where((mlat[0] >= lat_set[0]) & (mlat[0] <lat_set[1]))

df_CDF_6070 = [] 

for i in s6070[0]:
    # make a dataframe of the relevant data
    data_df = {'dbn_nez': Bn[:,i], 'dbe_nez':Be[:,i], 'MLT':mlt[:,i]}
    # append the dataframe to a list
    df_CDF_6070.append(pd.DataFrame(data=data_df, index=date_UTC).rename_axis(index='Date_UTC'))

# choose the latitudes to use
if lat_set == (lat_set[0],lat_set[1]):
    df_CDF = df_CDF_6070

In [10]:
# ULF power from netCDF data

# timestamps to divide data into days
time = df_CDF[0].index
start = time[0]
timerange = len(time)/(60*24)
dt=60.0
end = start + datetime.timedelta(days=timerange)
indices = [start + datetime.timedelta(days=i) for i in range(int(timerange))]

dfs = pd.DataFrame()
dfs_narrow = pd.DataFrame()

for df in df_CDF:

    Bh = component_funcs[component](df)
    
    for date in indices:
        if date == start: # adds 2 hours of data to the end
            s = date
            e = date + datetime.timedelta(days=1, hours=2)

        elif date == end -datetime.timedelta(days=1): # adds two hours of data to the beginning
            s = date - datetime.timedelta(hours=2)
            e = end

        else: # adds two hours of data to both sides
            s = date -datetime.timedelta(hours=2)
            e = date + datetime.timedelta(days=1, hours=2)

        # take the day in question (plus the overlaps)
        df1 = df[(df.index >= s) & (df.index < e)]
        part = df1.index # times
        
        # get the component and make a series
        B = component_funcs[component](df1)
        ser = pd.Series(data=B, index=part)

        # maximum count of consecutive nans
        nans = ser.isnull().astype(int).groupby(ser.notnull().astype(int).cumsum()).sum().max()

        # if there is a datagap larger than 2 minutes or the data starts with zero, do not include
        if nans > 2 or ser.isna().iloc[0] == True:
            continue

        # interpolate for possible nans
        ser = ser.interpolate()
        B = ser.to_numpy()

        # get the wavepower
        power, freqs, scales, pc5_scale_avg,coi = pc5_wave(B, dt)

        # pc5 power
        dat = {'Pc5_scale_avg':pc5_scale_avg, 'MLT' : df1['MLT'].to_numpy()}
        df2 = pd.DataFrame(data=dat, index=part)
        df2 =df2[(df2.index>= date) & (df2.index < date+ datetime.timedelta(days=1))] #remove the added hours
        dfs=pd.concat([dfs, df2])

In [11]:
# merge all dataframes together and calculate the mean pc5 power
global_ULF = dfs.groupby('Date_UTC').mean()
globalstat = global_ULF.groupby('Date_UTC').size()
time = global_ULF.index

In [ ]:
# divide data into different MLT sectors

idx = pd.date_range(min(time), max(time), freq='min')

# ULF power in the nightside
MLT1806 = dfs[((dfs['MLT']>= 18) & (dfs['MLT'] <= 24)) | ((dfs['MLT']>= 0) & (dfs['MLT'] < 6))]
night_pc5 = MLT1806.groupby('Date_UTC')['Pc5_scale_avg'].mean()
nightstat = MLT1806.groupby('Date_UTC').size()

# missing date --> NaN
if len(night_pc5) != len(time):
    night_pc5 = night_pc5.reindex(idx,fill_value=np.nan)
    nightstat = nightstat.reindex(idx, fill_value=np.nan)

# ULF power in the dayside
MLT0618 = dfs[(dfs['MLT'] >= 6) & (dfs['MLT'] < 18)]
day_pc5 = MLT0618.groupby('Date_UTC')['Pc5_scale_avg'].mean()
daystat = MLT0618.groupby('Date_UTC').size()

# missing date --> NaN
if len(day_pc5) != len(time):
    day_pc5 = day_pc5.reindex(idx,fill_value=np.nan)
    daystat = daystat.reindex(idx, fill_value=np.nan)

# ULF power in the midnight sector
MLT213 = dfs[((dfs['MLT']>= 21) & (dfs['MLT'] <= 24)) | ((dfs['MLT']>= 0) & (dfs['MLT'] < 3))]
midnight_pc5 = MLT213.groupby('Date_UTC')['Pc5_scale_avg'].mean()
midnightstat = MLT213.groupby('Date_UTC').size()

# missing date --> NaN
if len(midnight_pc5) != len(time):
    midnight_pc5 = midnight_pc5.reindex(idx, fill_value=np.nan)
    midnightstat = midnightstat.reindex(idx, fill_value=np.nan)

# ULF power in the dawn sector
MLT39 = dfs[(dfs['MLT'] >= 3) & (dfs['MLT'] < 9)]
dawn_pc5 = MLT39.groupby('Date_UTC')['Pc5_scale_avg'].mean()
dawnstat = MLT39.groupby('Date_UTC').size()

# missing date --> NaN
if len(dawn_pc5) != len(time):
    dawn_pc5 = dawn_pc5.reindex(idx, fill_value=np.nan)
    dawnstat = dawnstat.reindex(idx, fill_value=np.nan)

# ULF power in the day sector
MLT915 = dfs[(dfs['MLT'] >= 9) & (dfs['MLT'] < 15)]
noon_pc5 = MLT915.groupby('Date_UTC')['Pc5_scale_avg'].mean()
noonstat = MLT915.groupby('Date_UTC').size()

# missing date --> NaN
if len(noon_pc5) != len(time):
    noon_pc5 = noon_pc5.reindex(idx, fill_value=np.nan)
    noonstat = noonstat.reindex(idx, fill_value=np.nan)

# ULF power in the dusk sector
MLT1521 = dfs[(dfs['MLT'] >= 15) & (dfs['MLT'] < 21)]
dusk_pc5 = MLT1521.groupby('Date_UTC')['Pc5_scale_avg'].mean()
duskstat = MLT1521.groupby('Date_UTC').size()

# missing date --> NaN
if len(dusk_pc5) != len(time):
    dusk_pc5 = dusk_pc5.reindex(idx, fill_value=np.nan)
    duskstat = duskstat.reindex(idx, fill_value=np.nan)


datas = {'date':global_ULF.index.date,'time':global_ULF.index.time,'Pfull':global_ULF['Pc5_scale_avg'].to_numpy(),'Nfull':globalstat.to_numpy(dtype=int),'Pdawn':dawn_pc5.to_numpy(),'Ndawn':dawnstat.to_numpy(dtype=int),'Pnoon':noon_pc5.to_numpy(),'Nnoon':noonstat.to_numpy(dtype=int),'Pdusk':dusk_pc5.to_numpy(),'Ndusk':duskstat.to_numpy(dtype=int),'Pmidnight':midnight_pc5.to_numpy(),'Nmidnight':midnightstat.to_numpy(dtype=int),'Pday':day_pc5.to_numpy(),'Nday':daystat.to_numpy(dtype=int),'Pnight':night_pc5.to_numpy(),'Nnight':nightstat.to_numpy(dtype=int)}

d = pd.DataFrame(data=datas)
d.index=time

In [ ]:
# filename properties

component_text = '' if component == 'H' else f'_{component}'

In [ ]:
#write into monthly files

width = 40

firstline = "#"+(width-2)*"*"+"#"+"\n"
headerline = "#"+8*" "+"Pc5 MLT-dependent index"+"\n"

for m in range(1,13):
    
    filepath = f'Pc5_MLTd_index_LAT_{lat_set[0]}_{lat_set[1]}_{YYYY}{m:02d}{component_text}.dat'

    monthdata = d[(pd.DatetimeIndex(d['date']).month == m)]
    with open(filepath,"w") as file:
        datastring = monthdata.to_string(index=False,na_rep='NaN')
        file.write(firstline)
        file.write(headerline)
        file.write("#"+(width-1)*"-"+"\n")
        file.write(f"# Date of creation: {datetime.date.today().strftime('%Y/%m/%d')}\n")
        file.write(f"# Mirja Ojuva, Veera Lipsanen, Sanni Hoilijoki, Lucile Turc\n# Website: \n# Github address: https://github.com/wavestorms/mlt-ulf\n")
        file.write(width*"#"+"\n# P: Pc5 index in a given sector\n# N: number of SuperMAG stations used to calculate the index\n")
        file.write("#"+(width-1)*"-"+"\n")
        file.write("# Definition of MLT sectors:\n# Day: 6-18 MLT\n# Night: 18-6 MLT\n# Dawn: 3-9 MLT\n# Noon: 9-15 MLT\n# Dusk: 15-21 MLT\n# Midnight: 21-3 MLT\n")
        file.write(width*"#"+"\n")
        file.write(datastring)